In [1]:
import os
import re

# 현재 폴더(transformer_v2)에 압축이 풀린 파일 경로
kor_path = "./korean-english-park.train.ko"
eng_path = "./korean-english-park.train.en"

# 1. 병렬 데이터 로드 및 중복 제거 함수
def clean_corpus(kor_path, eng_path):
    with open(kor_path, "r", encoding="utf-8") as f: 
        kor = f.read().splitlines()
    with open(eng_path, "r", encoding="utf-8") as f: 
        eng = f.read().splitlines()
    
    assert len(kor) == len(eng), "한국어와 영어 데이터의 줄 수가 다릅니다!"
    
    # 병렬 코퍼스의 짝을 유지하며 중복 제거 (탭으로 묶어서 set 적용 후 다시 분리)
    cleaned_corpus = list(set(["\t".join([k, e]) for k, e in zip(kor, eng)]))
    
    kor_clean, eng_clean = [], []
    for pair in cleaned_corpus:
        k, e = pair.split('\t')
        kor_clean.append(k)
        eng_clean.append(e)
        
    print(f"원본 데이터 개수: {len(kor)}개")
    print(f"중복 제거 후 데이터 개수: {len(kor_clean)}개")
    
    return kor_clean, eng_clean

# 2. 문장 내부 노이즈 정제 (정규식)
def preprocess_sentence(sentence):
    sentence = sentence.lower() # 소문자 변환
    # 알파벳, 한글, 문장부호(? . ! ,)만 남기고 모두 공백으로 치환
    sentence = re.sub(r"[^a-zA-Z가-힣?.!,]+", " ", sentence) 
    # 문장부호 양옆에 공백을 띄워 단어와 분리되게 함
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    # 여러 개의 공백을 하나로 압축
    sentence = re.sub(r'[" "]+', " ", sentence) 
    return sentence.strip()

# 로직 실행
kor_lines, eng_lines = clean_corpus(kor_path, eng_path)

kor_corpus = [preprocess_sentence(s) for s in kor_lines]
eng_corpus = [preprocess_sentence(s) for s in eng_lines]

print("\n✅ 데이터 정제 완료! 0번째 샘플 확인:")
print(f"KOR: {kor_corpus[0]}")
print(f"ENG: {eng_corpus[0]}")

원본 데이터 개수: 94123개
중복 제거 후 데이터 개수: 78968개

✅ 데이터 정제 완료! 0번째 샘플 확인:
KOR: 이들 업체는 만명 이상 거주하는 시골 사람들을 위해 메일 주문 판매부를 만들었다 .
ENG: whole mail order departments to more than , rural residents spread over an area twice the size of texas .


In [3]:
import sentencepiece as spm
import torch
from torch.nn.utils.rnn import pad_sequence

# 1. 단어 사전 생성을 위한 임시 파일 작성
with open('ko_corpus.txt', 'w', encoding='utf-8') as f:
    for sent in kor_corpus: f.write(f'{sent}\n')

with open('en_corpus.txt', 'w', encoding='utf-8') as f:
    for sent in eng_corpus: f.write(f'{sent}\n')

# 2. 토크나이저 훈련 함수
def generate_tokenizer(corpus_file, model_prefix, vocab_size=20000):
    spm.SentencePieceTrainer.Train(
        f'--input={corpus_file} --model_prefix={model_prefix} '
        f'--vocab_size={vocab_size} --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3'
    )
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load(f'{model_prefix}.model')
    return tokenizer

print("SentencePiece 모델 학습 중... ⏳")
ko_tokenizer = generate_tokenizer('ko_corpus.txt', 'ko_spm')
en_tokenizer = generate_tokenizer('en_corpus.txt', 'en_spm')

# 3. 토큰화, 길이 필터링, 그리고 BOS/EOS 수동 부착
def tokenize_and_pad(src_corpus, tgt_corpus, src_tokenizer, tgt_tokenizer, max_len=50):
    src_tokens, tgt_tokens = [], []
    
    # 토크나이저에서 정한 bos, eos 번호 가져오기 (지정해둔 1, 2번)
    bos_id = tgt_tokenizer.bos_id()
    eos_id = tgt_tokenizer.eos_id()
    
    for src, tgt in zip(src_corpus, tgt_corpus):
        # 텍스트를 숫자로 인코딩
        src_encoded = src_tokenizer.EncodeAsIds(src)
        tgt_encoded = tgt_tokenizer.EncodeAsIds(tgt)
        
        # 타겟 문장에 수동으로 [BOS] + 내용 + [EOS] 붙이기
        tgt_encoded_with_tags = [bos_id] + tgt_encoded + [eos_id]
        
        # 길이 제한 필터링 (max_len 50 이하만 통과)
        if len(src_encoded) <= max_len and len(tgt_encoded_with_tags) <= max_len:
            src_tokens.append(torch.tensor(src_encoded))
            tgt_tokens.append(torch.tensor(tgt_encoded_with_tags))
            
    # 4. 패딩 (길이를 맞추기 위해 0으로 채움)
    enc_train = pad_sequence(src_tokens, batch_first=True, padding_value=0)
    dec_train = pad_sequence(tgt_tokens, batch_first=True, padding_value=0)
    
    return enc_train, dec_train

enc_train, dec_train = tokenize_and_pad(kor_corpus, eng_corpus, ko_tokenizer, en_tokenizer, max_len=50)

print("\n✅ 토큰화 및 수동 BOS/EOS 부착 패딩 완료!")
print(f"인코더 입력(enc_train) 텐서 크기: {enc_train.shape}")
print(f"디코더 입력(dec_train) 텐서 크기: {dec_train.shape}")

SentencePiece 모델 학습 중... ⏳


I0000 00:00:1783861159.948836    5792 sentencepiece_trainer.cc:227] Running command: --input=ko_corpus.txt --model_prefix=ko_spm --vocab_size=20000 --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
I0000 00:00:1783861159.949905    5792 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: ko_corpus.txt
  input_format: 
  model_prefix: ko_spm
  model_type: UNIGRAM
  vocab_size: 20000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
 


✅ 토큰화 및 수동 BOS/EOS 부착 패딩 완료!
인코더 입력(enc_train) 텐서 크기: torch.Size([75567, 50])
디코더 입력(dec_train) 텐서 크기: torch.Size([75567, 50])


In [5]:
import torch
import torch.nn as nn
import math

# 1. 패딩 마스크 및 미래 토큰 마스크(Look-ahead) 생성 함수
def create_padding_mask(seq):
    return (seq == 0).unsqueeze(1).unsqueeze(2)

def create_look_ahead_mask(size, device):
    mask = torch.triu(torch.ones((size, size), device=device), diagonal=1).bool()
    return mask

# 2. 포지셔널 인코딩 (위치 정보 부여)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# 3. 트랜스포머 메인 클래스
class Transformer(nn.Module):
    def __init__(self, n_layers=2, d_model=512, n_heads=8, d_ff=2048, 
                 src_vocab_size=20000, tgt_vocab_size=20000, dropout=0.2):
        super(Transformer, self).__init__()
        
        self.src_emb = nn.Embedding(src_vocab_size, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        
        self.transformer = nn.Transformer(
            d_model=d_model, 
            nhead=n_heads, 
            num_encoder_layers=n_layers, 
            num_decoder_layers=n_layers, 
            dim_feedforward=d_ff, 
            dropout=dropout,
            batch_first=True
        )
        
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.fc_out.weight = self.tgt_emb.weight # 가중치 공유

    def forward(self, src, tgt):
        src_key_padding_mask = (src == 0)
        tgt_key_padding_mask = (tgt == 0)
        tgt_mask = self.transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        
        src_emb = self.pos_enc(self.src_emb(src) * math.sqrt(self.transformer.d_model))
        tgt_emb = self.pos_enc(self.tgt_emb(tgt) * math.sqrt(self.transformer.d_model))
        
        out = self.transformer(
            src=src_emb, tgt=tgt_emb,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
            tgt_mask=tgt_mask
        )
        
        return self.fc_out(out)

print("Transformer Architecture Defined Successfully!")

Transformer Architecture Defined Successfully!


In [6]:
# 1. 모델 인스턴스화 및 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 사용 중인 디바이스: {device}")

model = Transformer(
    n_layers=2, d_model=512, n_heads=8, d_ff=2048,
    src_vocab_size=20000, tgt_vocab_size=20000, dropout=0.2
).to(device)

# 2. 옵티마이저 (Adam: betas=(0.9, 0.98), eps=1e-9 제약사항)
optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)

# 3. 커스텀 학습률 스케줄러 (논문 스펙)
class LearningRateScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps=4000, last_epoch=-1):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super(LearningRateScheduler, self).__init__(optimizer, last_epoch)
        
    def get_lr(self):
        step = max(1, self.last_epoch)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        lr = (self.d_model ** -0.5) * min(arg1, arg2)
        return [lr for _ in self.base_lrs]

scheduler = LearningRateScheduler(optimizer, d_model=512)

# 4. 커스텀 Loss 함수 (0으로 패딩된 부분은 Loss 계산에서 제외)
loss_object = nn.CrossEntropyLoss(reduction='none')

def loss_function(real, pred):
    mask = (real != 0) # 정답이 0(패딩)이 아닌 위치만 True
    # 모델의 출력 형태 [Batch, Seq, Vocab] -> [Batch * Seq, Vocab]
    loss_ = loss_object(pred.reshape(-1, pred.size(-1)), real.reshape(-1))
    mask = mask.reshape(-1).float()
    loss_ *= mask # 패딩 자리는 loss를 0으로 만듦
    # 실제 마스킹되지 않은 토큰의 개수로 나누어 스케일링
    return loss_.sum() / mask.sum()

# 5. 번역 추론용 함수 (평가 시 사용)
def translate(sentence, model, src_tokenizer, tgt_tokenizer, max_len=50):
    model.eval() # 평가 모드 전환
    
    # 입력 문장 전처리 및 토큰화
    sent = preprocess_sentence(sentence)
    src_tokens = src_tokenizer.EncodeAsIds(sent)
    src_tensor = torch.tensor([src_tokens]).to(device)
    
    # 타겟 입력 초기화 (<BOS>만 넣은 상태로 시작)
    tgt_tokens = [tgt_tokenizer.bos_id()]
    
    for _ in range(max_len):
        tgt_tensor = torch.tensor([tgt_tokens]).to(device)
        
        with torch.no_grad():
            output = model(src_tensor, tgt_tensor)
            
        # 가장 확률이 높은 다음 단어 예측
        next_word_idx = output.argmax(dim=-1)[:, -1].item()
        tgt_tokens.append(next_word_idx)
        
        # <EOS>가 나오면 번역 종료
        if next_word_idx == tgt_tokenizer.eos_id():
            break
            
    # 앞뒤의 <BOS>, <EOS> 떼고 디코딩하여 문자열로 반환
    return tgt_tokenizer.DecodeIds(tgt_tokens[1:-1])

print("Setup for Optimizer, Scheduler, Loss, and Translation logic is Complete!")

현재 사용 중인 디바이스: cuda
Setup for Optimizer, Scheduler, Loss, and Translation logic is Complete!


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# 1. 하이퍼파라미터 및 데이터로더 설정
batch_size = 64
epochs = 10 # 훈련 횟수 (결과를 보고 싶다면 10~20 추천)

dataset = TensorDataset(enc_train, dec_train)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 2. 필수 번역 테스트 예문
test_sentences = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

print("====== 🚀 한-영 번역기 훈련 시작 ======")
for epoch in range(epochs):
    model.train() # 훈련 모드 전환
    total_loss = 0
    
    for batch_idx, (src_batch, tgt_batch) in enumerate(dataloader):
        src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)
        
        # 디코더 입력: 마지막 토큰(<EOS>) 제외
        tgt_input = tgt_batch[:, :-1] 
        # 디코더 정답: 첫 번째 토큰(<BOS>) 제외
        tgt_real = tgt_batch[:, 1:]   
        
        optimizer.zero_grad() # 기울기 초기화
        output = model(src_batch, tgt_input) # 모델 예측
        
        loss = loss_function(tgt_real, output) # Loss 계산
        loss.backward() # 역전파
        
        optimizer.step() # 가중치 업데이트
        scheduler.step() # 학습률 스텝 진행 (매 배치마다)
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    print(f"\n[Epoch {epoch+1}/{epochs}] Average Loss: {avg_loss:.4f}")
    
    # 3. 매 에폭마다 번역 성능 실시간 확인
    print("-" * 50)
    for sent in test_sentences:
        translated = translate(sent, model, ko_tokenizer, en_tokenizer)
        print(f"Korean : {sent}")
        print(f"English: {translated}")
    print("-" * 50)
    
print("====== ✨ 훈련이 성공적으로 완료되었습니다! ======")

====== 🚀 한-영 번역기 훈련 시작 ======


In [ ]:
print("====== 💡 실전 번역기 테스트 시작! ======")
print("(번역기를 종료하시려면 '0'을 입력하세요)\n")

while True:
    user_input = input("🇰🇷 한국어 문장을 입력하세요: ")
    
    # 종료 조건
    if user_input == '0':
        print("번역기를 종료합니다. 수고하셨습니다!")
        break
        
    # 빈 입력 방지
    if not user_input.strip():
        continue
        
    # 번역 수행
    try:
        translated_sentence = translate(user_input, model, ko_tokenizer, en_tokenizer)
        print(f"🇺🇸 번역 결과: {translated_sentence}\n")
    except Exception as e:
        print(f"에러가 발생했습니다: {e}\n")